# FraudGuard - Transaction Fraud Intelligence
## Notebook 2 - Feature Engineering

### PACE: Construct - Feature Engineering

**Objective:** Build model-ready features from raw transaction data.

**3 Engineering Tasks**  
1. Velocity features - transaction frequency signals
2. Categorical encoding - convert strings to numbers
3. Feature selection - reduce 666 columns to focussed set

**Key Principle:** All features must be available at the moment of transaction - no post-transaction leakage.

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

# Load raw data
df_trans = pd.read_csv('../data/train_transaction.csv')
df_identity = pd.read_csv('../data/train_identity.csv')

# Merge
df = df_trans.merge(df_identity, on = 'TransactionID', how ='left')

print(f"Full dataset loaded")
print(f"Shape: {df.shape}")
print(f"Fraud rate: {df['isFraud'].mean()*100:.3f}%")

Full dataset loaded
Shape: (590540, 434)
Fraud rate: 3.499%


In [2]:
# Confirm clean state
assert df.shape == (590540, 434), \
    f"Unexpected shape: {df.shape}. Restart kernel."
print(f"Shape confirmed: {df.shape}")

# Calculate missing percentages
missing_pct = (df.isnull().sum(axis=0) / len(df) * 100)

# Three tiers
tier1_drop = missing_pct[missing_pct > 99].index.tolist()
tier2_indicator = missing_pct[
    (missing_pct >= 50) & (missing_pct <= 99)
].index.tolist()
tier3_impute = missing_pct[
    (missing_pct > 0) & (missing_pct < 50)
].index.tolist()

print(f"Tier 1 — Drop:      {len(tier1_drop)} columns")
print(f"Tier 2 — Indicator: {len(tier2_indicator)} columns")
print(f"Tier 3 — Impute:    {len(tier3_impute)} columns")

# Step 1 — Drop Tier 1
df = df.drop(columns=tier1_drop)
print(f"\nAfter dropping Tier 1: {df.shape}")

# Step 2 — Create binary indicators for Tier 2
for col in tier2_indicator:
    df[f'{col}_missing'] = df[col].isnull().astype(int)
print(f"After adding indicators: {df.shape}")

# Step 3 — Impute Tier 2 and Tier 3
numeric_missing = [c for c in tier2_indicator + tier3_impute
                   if df[c].dtype != 'object']
cat_missing = [c for c in tier2_indicator + tier3_impute
               if df[c].dtype == 'object']

print(f"\nNumeric columns to impute: {len(numeric_missing)}")
print(f"Categorical columns to impute: {len(cat_missing)}")

for col in numeric_missing:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_missing:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)

# Confirm
remaining = df.isnull().sum().sum()
print(f"\nMissing values remaining: {remaining}")
print(f"Final dataset shape: {df.shape}")

Shape confirmed: (590540, 434)
Tier 1 — Drop:      9 columns
Tier 2 — Indicator: 205 columns
Tier 3 — Impute:    200 columns

After dropping Tier 1: (590540, 425)
After adding indicators: (590540, 630)

Numeric columns to impute: 377
Categorical columns to impute: 28

Missing values remaining: 0
Final dataset shape: (590540, 630)


### Step 2 - Time Feature Engineering  
TransactionDT is seconds elapsed from reference point. Extracting: hour of day, day of week, time since first transaction. Time patterns are strong fraud signals - fraudsters operate at unusual hours and in rapid bursts.

In [3]:
# Extract time features from transactionDt
# Reference point unknown - relative patterns still valid

# Hour of day (0-23)
df['tx_hour'] = (df['TransactionDT']//3600)% 24

# Day of week (0-6)
df['tx_day'] = (df['TransactionDT']//(3600*24))% 7

# Days elapsed since start of dataset
df['tx_day_elapsed'] = df['TransactionDT']//(3600*24)

# Is it night transaction(11pm - 6am)
df['is_night'] = ((df['tx_hour'] >= 23) | (df['tx_hour'] <= 6)).astype(int)

# Is it a weekend?
df['is_weekend'] = (df['tx_day'] >= 5).astype(int)

print("Time features created:")
print(f" tx_hour:          {df['tx_hour'].nunique()} unique values")
print(f" tx_day:           {df['tx_day'].nunique()} unique values")
print(f" tx_day_elapsed:   {df['tx_day_elapsed'].nunique()} unique values(span of dataset)")
print(f" is_night:         {df['is_night'].mean()*100:.1f}% night transactions")
print(f" is_weekend:       {df['is_weekend'].mean()*100:.1f}% weekend transactions")

# Validate against EDA findings
print(f"\nFraud Rate by is_night:")
print(df.groupby('is_night')['isFraud'].mean()*100)
print(f"\nFraud Rate by is_weekend:")
print(df.groupby('is_weekend')['isFraud'].mean()*100)
print(f"\nDataset Shape: {df.shape}")

Time features created:
 tx_hour:          24 unique values
 tx_day:           7 unique values
 tx_day_elapsed:   182 unique values(span of dataset)
 is_night:         31.9% night transactions
 is_weekend:       28.8% weekend transactions

Fraud Rate by is_night:
is_night
0    3.297688
1    3.927962
Name: isFraud, dtype: float64

Fraud Rate by is_weekend:
is_weekend
0    3.547835
1    3.378366
Name: isFraud, dtype: float64

Dataset Shape: (590540, 635)


### Step 3 - Velocity Features  
Velocity measures transaction frequency per card over rolling time windows. Rapid transactions bursts signal fraud - a stolen card is used quickly before cancellation.  

Features:  
- Transactions per card in last hour
- Transactions per card in last day
- Mean transaction amount per card (spending baseline)
- Deviation from card's own baseline amount

In [4]:
# Sort by card and time for rolling calculations
df = df.sort_values(['card1', 'TransactionDT']).reset_index(drop = True)
print("Calculating velocity features...")
print("(This may take 1-2 minutes on full dataset)")

# Transaction per card total(proxy for card activity level)
card_tx_count = df.groupby('card1')['TransactionID'].transform('count')
df['card_tx_count'] = card_tx_count

# Mean transaction amount per card (spending baseline)
card_mean_amt = df.groupby('card1')['TransactionAmt'].transform('mean')
df['card_mean_amt'] = card_mean_amt

# Deviation from card's own baseline
df['amt_deviation'] = (df['TransactionAmt'] - df['card_mean_amt'])/(df['card_mean_amt']+1)

# Transaction amount ratio (tx vs card mean)
df['amt_ratio'] = df['TransactionAmt']/(df['card_mean_amt']+1)
print("Velocity features created:")
print(f" card_tx_count: total transactions per card")
print(f" car_mean_amt: mean spend per card")
print(f" amt_deviation: deviation from card baseline")
print(f" amt_ratio: this amount vs card mean")

# Validate  - do these features differ between fraud/legit?
print(f"\nFeature means by fraud label:")
vel_cols = ['card_tx_count', 'card_mean_amt', 'amt_deviation', 'amt_ratio']
for col in vel_cols:
    legit_mean = df[df['isFraud']==0][col].mean()
    fraud_mean = df[df['isFraud']==1][col].mean()
    print(f" {col:<20}"
          f"Legit: {legit_mean: >8.2f}"
          f"Fraud: {fraud_mean: >8.2f}")

print(f"\nDataset Shape: {df.shape}")

Calculating velocity features...
(This may take 1-2 minutes on full dataset)
Velocity features created:
 card_tx_count: total transactions per card
 car_mean_amt: mean spend per card
 amt_deviation: deviation from card baseline
 amt_ratio: this amount vs card mean

Feature means by fraud label:
 card_tx_count       Legit:  2535.85Fraud:  2334.68
 card_mean_amt       Legit:   135.36Fraud:   125.73
 amt_deviation       Legit:    -0.01Fraud:     0.18
 amt_ratio           Legit:     0.98Fraud:     1.17

Dataset Shape: (590540, 639)


### Step 4 - Categorical Encoding  
Model requires numerical inputs.  
Strategy:  
- Low cardinality (<10 categories): Label Encoding  
- High cardinality (>=10 categories): Frequency Encoding *(replace category with its frequency in dataset)*  

Frequency encoding chosen over one-hot for high cardinality - avoids creating hundreds of sparse columns

In [5]:
from sklearn.preprocessing import LabelEncoder

# identify categorical columns in current dataset
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns to encode: {len(cat_cols)}")

le = LabelEncoder()
for col in cat_cols:
    n_unique = df[col].nunique()

    if n_unique < 10: # Label encoding for low cardinality
        df[col] = le.fit_transform(df[col].astype(str))
        print(f" Label encoded: {col}({n_unique} categories)")
    else: # Frequency encoding for high cardinality
        freq_map = df[col].value_counts().to_dict()
        df[col] = df[col].map(freq_map)
        print(f" Frequency encoded: {col}({n_unique} categories)")

print(f"\nFinal dataset shape: {df.shape}")
print(f"Remaining categorical columns:"
      f"{len(df.select_dtypes(include=['object']).columns)}")

Categorical columns to encode: 29
 Label encoded: ProductCD(5 categories)
 Label encoded: card4(4 categories)
 Label encoded: card6(4 categories)
 Frequency encoded: P_emaildomain(59 categories)
 Frequency encoded: R_emaildomain(60 categories)
 Label encoded: M1(2 categories)
 Label encoded: M2(2 categories)
 Label encoded: M3(2 categories)
 Label encoded: M4(3 categories)
 Label encoded: M5(2 categories)
 Label encoded: M6(2 categories)
 Label encoded: M7(2 categories)
 Label encoded: M8(2 categories)
 Label encoded: M9(2 categories)
 Label encoded: id_12(2 categories)
 Label encoded: id_15(3 categories)
 Label encoded: id_16(2 categories)
 Label encoded: id_28(2 categories)
 Label encoded: id_29(2 categories)
 Frequency encoded: id_30(75 categories)
 Frequency encoded: id_31(130 categories)
 Frequency encoded: id_33(260 categories)
 Label encoded: id_34(4 categories)
 Label encoded: id_35(2 categories)
 Label encoded: id_36(2 categories)
 Label encoded: id_37(2 categories)
 Label enc

### Step 5 - Feature Selection  
Removing columns that add noise without signal:  
1. ID columns - nop predictive value
2. Raw TransactionDT - replaced by engineered time features
3. Columns with near zero variances  

**Retaining columns:** *transaction features, card features, velocity features, time features, missingness indicators*

In [6]:
# Explicit feature selection based on domain knowledge and EDA
# More defensible than automated variance filtering

feature_cols = [
    # Transaction characteristics
    'TransactionAmt',       # EDA Finding 1 — amount differs fraud/legit
    'ProductCD',            # EDA Finding 2 — CNP highest fraud rate

    # Card features
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',

    # Address match features
    'addr1', 'addr2',

    # Time features (engineered)
    'tx_hour', 'tx_day', 'tx_day_elapsed',
    'is_night', 'is_weekend',

    # Velocity features (engineered)
    'card_tx_count', 'card_mean_amt',
    'amt_deviation', 'amt_ratio',

    # Email domain features
    'P_emaildomain', 'R_emaildomain',

    # Distance features
    'dist1',

    # Count features (C columns)
    'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8',
    'C9', 'C10', 'C11', 'C12', 'C13', 'C14',

    # Time delta features (D columns — meaningful ones)
    'D1', 'D2', 'D3', 'D4', 'D10', 'D15',

    # Match flag features (M columns)
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6',

    # Identity features
    'id_01', 'id_02', 'id_03', 'id_05', 'id_06',
    'id_09', 'id_10', 'id_11', 'id_13', 'id_17',
    'id_19', 'id_20',

    # Device features
    'DeviceType',
]

# Keep only features that exist in df
feature_cols = [f for f in feature_cols if f in df.columns]
target_col = 'isFraud'

print(f"Features selected: {len(feature_cols)}")
print(f"\nFeature list:")
for f in feature_cols:
    print(f"  {f}")

# Create model-ready dataset
df_model = df[feature_cols + [target_col]].copy()

print(f"\nModel dataset shape: {df_model.shape}")
print(f"Fraud rate: {df_model[target_col].mean()*100:.3f}%")
print(f"Missing values: {df_model.isnull().sum().sum()}")

Features selected: 60

Feature list:
  TransactionAmt
  ProductCD
  card1
  card2
  card3
  card4
  card5
  card6
  addr1
  addr2
  tx_hour
  tx_day
  tx_day_elapsed
  is_night
  is_weekend
  card_tx_count
  card_mean_amt
  amt_deviation
  amt_ratio
  P_emaildomain
  R_emaildomain
  dist1
  C1
  C2
  C4
  C5
  C6
  C7
  C8
  C9
  C10
  C11
  C12
  C13
  C14
  D1
  D2
  D3
  D4
  D10
  D15
  M1
  M2
  M3
  M4
  M5
  M6
  id_01
  id_02
  id_03
  id_05
  id_06
  id_09
  id_10
  id_11
  id_13
  id_17
  id_19
  id_20
  DeviceType

Model dataset shape: (590540, 61)
Fraud rate: 3.499%
Missing values: 0


In [7]:
# Save model-ready dataset
df_model.to_csv('../data/fraud_clean.csv', index=False)
print(f"Saved: fraud_clean.csv")
print(f"Shape: {df_model.shape}")
print(f"Size: ~{df_model.memory_usage().sum() / 1024**2:.0f} MB")

Saved: fraud_clean.csv
Shape: (590540, 61)
Size: ~275 MB


In [8]:
# Sample for Streamlit Cloud deployment
# Full dataset too large for GitHub (275MB)
df_sample = df_model.sample(n=50000, random_state=42)
df_sample.to_csv('../data/fraud_sample.csv', index=False)

print(f"Sample saved: {df_sample.shape}")
print(f"Fraud rate: {df_sample['isFraud'].mean()*100:.3f}%")

Sample saved: (50000, 61)
Fraud rate: 3.402%
